In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.linalg import expm
from itertools import combinations, permutations, product
from pathlib import Path
from tqdm.auto import tqdm


# Random seed
seed = 20260804
rng = np.random.default_rng(seed)


# Numerical tolerance
tol = 1e-10


# Display
np.set_printoptions(
    precision=6,
    suppress=True
)


# Result folder
result_dir = Path("results")
result_dir.mkdir(exist_ok=True)

print("Random seed:", seed)
print("Result folder:", result_dir.resolve())

Random seed: 20260804
Result folder: C:\Users\liu.xuanc\Desktop\Code\Quantum-HOC\results


In [8]:
# Pauli matrices

I2 = np.eye(2, dtype=complex)

X = np.array([
    [0, 1],
    [1, 0]
], dtype=complex)

Y = np.array([
    [0, -1j],
    [1j, 0]
], dtype=complex)

Z = np.array([
    [1, 0],
    [0, -1]
], dtype=complex)

pauli = {
    "x": X,
    "y": Y,
    "z": Z
}


# Tensor product of a list of operators

def kron_all(operators):
    result = operators[0]

    for op in operators[1:]:
        result = np.kron(result, op)

    return result


# Put a single-qubit operator on site i

def one_body(op, i, N):
    operators = [I2] * N
    operators[i] = op

    return kron_all(operators)


# Put a product operator op_i ⊗ op_j on sites i and j

def two_body(op_i, op_j, i, j, N):
    operators = [I2] * N
    operators[i] = op_i
    operators[j] = op_j

    return kron_all(operators)


# Commutator

def commutator(A, B):
    return A @ B - B @ A

In [9]:
# Basic checks

print("Pauli commutator check:")
print("||[X,Y] - 2iZ|| =", np.linalg.norm(
    commutator(X, Y) - 2j * Z
))


N = 3

X0 = one_body(X, 0, N)
Z2 = one_body(Z, 2, N)
X0Z2 = two_body(X, Z, 0, 2, N)

print()
print("Three-qubit operator dimension:", X0.shape)
print("||X0 Z2 - embedded X0Z2|| =", np.linalg.norm(
    X0 @ Z2 - X0Z2
))

print("Hermiticity check:", np.linalg.norm(
    X0Z2 - X0Z2.conj().T
))

Pauli commutator check:
||[X,Y] - 2iZ|| = 0.0

Three-qubit operator dimension: (8, 8)
||X0 Z2 - embedded X0Z2|| = 0.0
Hermiticity check: 0.0


In [10]:
# Three physical nodes

N = 3
A, B, C = 0, 1, 2


# Pairwise XYZ Hamiltonian

def edge_hamiltonian(i, j, J, N):
    """
    H_ij = Jx X_i X_j + Jy Y_i Y_j + Jz Z_i Z_j
    """

    Jx, Jy, Jz = J

    H = (
        Jx * two_body(X, X, i, j, N)
        + Jy * two_body(Y, Y, i, j, N)
        + Jz * two_body(Z, Z, i, j, N)
    )

    return H


# Fixed couplings for the first test

J_AB = (1.0, 0.8, 1.2)
J_BC = (0.9, 1.1, 0.7)
J_AC = (1.2, 0.6, 1.0)


# Edge Hamiltonians

H_AB = edge_hamiltonian(A, B, J_AB, N)
H_BC = edge_hamiltonian(B, C, J_BC, N)
H_AC = edge_hamiltonian(A, C, J_AC, N)


# Total triangle Hamiltonian

H_triangle = H_AB + H_BC + H_AC

In [11]:
print("Hermiticity checks")

print("H_AB:", np.linalg.norm(H_AB - H_AB.conj().T))
print("H_BC:", np.linalg.norm(H_BC - H_BC.conj().T))
print("H_AC:", np.linalg.norm(H_AC - H_AC.conj().T))
print("H_triangle:", np.linalg.norm(
    H_triangle - H_triangle.conj().T
))


print()
print("Hamiltonian norms")

print("||H_AB|| =", np.linalg.norm(H_AB))
print("||H_BC|| =", np.linalg.norm(H_BC))
print("||H_AC|| =", np.linalg.norm(H_AC))
print("||H_triangle|| =", np.linalg.norm(H_triangle))


print()
print("Pairwise commutator norms")

print("||[H_AB, H_BC]|| =", np.linalg.norm(
    commutator(H_AB, H_BC)
))

print("||[H_AB, H_AC]|| =", np.linalg.norm(
    commutator(H_AB, H_AC)
))

print("||[H_BC, H_AC]|| =", np.linalg.norm(
    commutator(H_BC, H_AC)
))

Hermiticity checks
H_AB: 0.0
H_BC: 0.0
H_AC: 0.0
H_triangle: 0.0

Hamiltonian norms
||H_AB|| = 4.963869458396342
||H_BC|| = 4.4810713004816165
||H_AC|| = 4.732863826479693
||H_triangle|| = 8.192679659305616

Pairwise commutator norms
||[H_AB, H_BC]|| = 13.194908108812276
||[H_AB, H_AC]|| = 13.282891251531044
||[H_BC, H_AC]|| = 12.567895607459509


In [12]:
# Complete single-qubit Pauli basis

pauli_basis = {
    "I": I2,
    "X": X,
    "Y": Y,
    "Z": Z
}


# Decompose an operator into Pauli strings

def pauli_decomposition(operator, N, tol=1e-10):

    rows = []

    for labels in product("IXYZ", repeat=N):

        P = kron_all([
            pauli_basis[label]
            for label in labels
        ])

        coefficient = np.trace(P @ operator) / (2**N)

        if abs(coefficient) > tol:

            rows.append({
                "pauli_string": "".join(labels),
                "coefficient_real": coefficient.real,
                "coefficient_imag": coefficient.imag,
                "support_size": sum(
                    label != "I"
                    for label in labels
                )
            })

    return pd.DataFrame(rows)


# Hermitian commutator-generated operators

K_AB_BC = -1j * commutator(H_AB, H_BC)
K_AB_AC = -1j * commutator(H_AB, H_AC)
K_BC_AC = -1j * commutator(H_BC, H_AC)


operators = {
    "-i[H_AB,H_BC]": K_AB_BC,
    "-i[H_AB,H_AC]": K_AB_AC,
    "-i[H_BC,H_AC]": K_BC_AC
}


all_terms = []

for name, operator in operators.items():

    df = pauli_decomposition(operator, N)
    df.insert(0, "operator", name)

    all_terms.append(df)

    print(name)
    print("Hermiticity error:",
          np.linalg.norm(operator - operator.conj().T))
    print("Number of Pauli strings:", len(df))
    print("Support sizes:", sorted(df["support_size"].unique()))

    display(df)

    print()

-i[H_AB,H_BC]
Hermiticity error: 0.0
Number of Pauli strings: 6
Support sizes: [np.int64(3)]


,operator,pauli_string,coefficient_real,coefficient_imag,support_size
0,"-i[H_AB,H_BC]",XYZ,-1.40,0.0,3
1,"-i[H_AB,H_BC]",XZY,2.20,0.0,3
2,"-i[H_AB,H_BC]",YXZ,1.12,0.0,3
3,"-i[H_AB,H_BC]",YZX,-1.44,0.0,3
4,"-i[H_AB,H_BC]",ZXY,-2.64,0.0,3
5,"-i[H_AB,H_BC]",ZYX,2.16,0.0,3



-i[H_AB,H_AC]
Hermiticity error: 0.0
Number of Pauli strings: 6
Support sizes: [np.int64(3)]


,operator,pauli_string,coefficient_real,coefficient_imag,support_size
0,"-i[H_AB,H_AC]",XYZ,1.60,0.0,3
1,"-i[H_AB,H_AC]",XZY,-1.44,0.0,3
2,"-i[H_AB,H_AC]",YXZ,-2.00,0.0,3
3,"-i[H_AB,H_AC]",YZX,2.88,0.0,3
4,"-i[H_AB,H_AC]",ZXY,1.20,0.0,3
5,"-i[H_AB,H_AC]",ZYX,-1.92,0.0,3



-i[H_BC,H_AC]
Hermiticity error: 0.0
Number of Pauli strings: 6
Support sizes: [np.int64(3)]


,operator,pauli_string,coefficient_real,coefficient_imag,support_size
0,"-i[H_BC,H_AC]",XYZ,-2.64,0.0,3
1,"-i[H_BC,H_AC]",XZY,1.68,0.0,3
2,"-i[H_BC,H_AC]",YXZ,1.08,0.0,3
3,"-i[H_BC,H_AC]",YZX,-0.84,0.0,3
4,"-i[H_BC,H_AC]",ZXY,-1.80,0.0,3
5,"-i[H_BC,H_AC]",ZYX,2.20,0.0,3


In [13]:
commutator_support = pd.concat(
    all_terms,
    ignore_index=True
)

commutator_support.to_csv(
    result_dir / "commutator_pauli_support.csv",
    index=False
)

In [14]:
# Jordan product

def jordan(A, B):
    return 0.5 * (A @ B + B @ A)


# Jordan associator

def jordan_associator(A, B, C):
    """
    (A o B) o C - A o (B o C)
    = 1/4 [B, [A, C]]
    """

    return (
        jordan(jordan(A, B), C)
        - jordan(A, jordan(B, C))
    )

In [15]:
A_outer_AB = jordan_associator(
    H_AC, H_AB, H_BC
)

A_outer_BC = jordan_associator(
    H_AB, H_BC, H_AC
)

A_outer_AC = jordan_associator(
    H_AB, H_AC, H_BC
)


associators = {
    "A_outer_AB": (
        A_outer_AB,
        0.25 * commutator(
            H_AB,
            commutator(H_AC, H_BC)
        )
    ),

    "A_outer_BC": (
        A_outer_BC,
        0.25 * commutator(
            H_BC,
            commutator(H_AB, H_AC)
        )
    ),

    "A_outer_AC": (
        A_outer_AC,
        0.25 * commutator(
            H_AC,
            commutator(H_AB, H_BC)
        )
    )
}

In [16]:
associator_terms = []
associator_summary = []

for name, (A_jordan, A_nested) in associators.items():

    df = pauli_decomposition(A_jordan, N)
    df.insert(0, "operator", name)

    associator_terms.append(df)

    identity_error = np.linalg.norm(
        A_jordan - A_nested
    )

    hermiticity_error = np.linalg.norm(
        A_jordan - A_jordan.conj().T
    )

    associator_summary.append({
        "operator": name,
        "norm": np.linalg.norm(A_jordan),
        "identity_error": identity_error,
        "hermiticity_error": hermiticity_error,
        "number_of_terms": len(df),
        "support_sizes": str(
            sorted(df["support_size"].unique())
        )
    })

    print(name)
    print("Norm:", np.linalg.norm(A_jordan))
    print("Jordan identity error:", identity_error)
    print("Hermiticity error:", hermiticity_error)
    print("Support sizes:",
          sorted(df["support_size"].unique()))

    display(df)

    print()

A_outer_AB
Norm: 11.86638141979264
Jordan identity error: 1.0934428697813141e-15
Hermiticity error: 1.5700924586837752e-16
Support sizes: [np.int64(2)]


,operator,pauli_string,coefficient_real,coefficient_imag,support_size
0,A_outer_AB,IXX,-1.656,0.0,2
1,A_outer_AB,IYY,-1.920,0.0,2
2,A_outer_AB,IZZ,-1.752,0.0,2
3,A_outer_AB,XIX,1.384,0.0,2
4,A_outer_AB,YIY,1.908,0.0,2
5,A_outer_AB,ZIZ,1.596,0.0,2



A_outer_BC
Norm: 11.397414443635888
Jordan identity error: 9.023764073195417e-16
Hermiticity error: 1.5700924586837752e-16
Support sizes: [np.int64(2)]


,operator,pauli_string,coefficient_real,coefficient_imag,support_size
0,A_outer_BC,XIX,-1.384,0.0,2
1,A_outer_BC,XXI,1.352,0.0,2
2,A_outer_BC,YIY,-1.908,0.0,2
3,A_outer_BC,YYI,1.996,0.0,2
4,A_outer_BC,ZIZ,-1.596,0.0,2
5,A_outer_BC,ZZI,1.524,0.0,2



A_outer_AC
Norm: 11.876947756052477
Jordan identity error: 1.4432899320127035e-15
Hermiticity error: 0.0
Support sizes: [np.int64(2)]


,operator,pauli_string,coefficient_real,coefficient_imag,support_size
0,A_outer_AC,IXX,-1.656,0.0,2
1,A_outer_AC,IYY,-1.920,0.0,2
2,A_outer_AC,IZZ,-1.752,0.0,2
3,A_outer_AC,XXI,1.352,0.0,2
4,A_outer_AC,YYI,1.996,0.0,2
5,A_outer_AC,ZZI,1.524,0.0,2


In [17]:
associator_summary = pd.DataFrame(
    associator_summary
)

associator_terms = pd.concat(
    associator_terms,
    ignore_index=True
)

display(associator_summary)

associator_summary.to_csv(
    result_dir / "associator_summary.csv",
    index=False
)

associator_terms.to_csv(
    result_dir / "associator_pauli_support.csv",
    index=False
)

,operator,norm,identity_error,hermiticity_error,number_of_terms,support_sizes
0,A_outer_AB,11.866381,1.093443e-15,1.570092e-16,6,[np.int64(2)]
1,A_outer_BC,11.397414,9.023764e-16,1.570092e-16,6,[np.int64(2)]
2,A_outer_AC,11.876948,1.443290e-15,0.000000e+00,6,[np.int64(2)]


Cycle, path, star and repeated-edge motifs

In [18]:
# Four-node system

N4 = 4
A, B, C, D = 0, 1, 2, 3


# Additional couplings

J_CD = (0.75, 1.25, 0.95)
J_AD = (1.15, 0.85, 0.65)


# Four-qubit edge Hamiltonians

H4_AB = edge_hamiltonian(A, B, J_AB, N4)
H4_BC = edge_hamiltonian(B, C, J_BC, N4)
H4_AC = edge_hamiltonian(A, C, J_AC, N4)

H4_CD = edge_hamiltonian(C, D, J_CD, N4)
H4_AD = edge_hamiltonian(A, D, J_AD, N4)


# Three-edge nested commutator

def three_edge_operator(H_inner_1, H_inner_2, H_outer):
    """
    Nested word:
        1/4 [H_outer, [H_inner_1, H_inner_2]]

    This is an algebraic bracketing convention,
    not a temporal protocol.
    """

    return 0.25 * commutator(
        H_outer,
        commutator(H_inner_1, H_inner_2)
    )


# --------------------------------------------------
# Four motif classes
# --------------------------------------------------

# Triangle: AB - BC - AC
M_cycle = three_edge_operator(
    H4_AB, H4_BC, H4_AC
)

# Open path: AB - BC - CD
M_path = three_edge_operator(
    H4_AB, H4_BC, H4_CD
)

# Three-edge star centred at A: AB - AC - AD
M_star = three_edge_operator(
    H4_AB, H4_AC, H4_AD
)

# Repeated edge: AB - BC - AB
M_repeated = three_edge_operator(
    H4_AB, H4_BC, H4_AB
)


motifs = {
    "cycle_AB_BC_AC": M_cycle,
    "path_AB_BC_CD": M_path,
    "star_AB_AC_AD": M_star,
    "repeated_AB_BC_AB": M_repeated
}

In [19]:
motif_summary = []
motif_terms = []

for name, operator in motifs.items():

    df = pauli_decomposition(operator, N4)
    df.insert(0, "motif", name)

    motif_terms.append(df)

    support_sizes = sorted(
        int(x) for x in df["support_size"].unique()
    )

    motif_summary.append({
        "motif": name,
        "norm": np.linalg.norm(operator),
        "hermiticity_error": np.linalg.norm(
            operator - operator.conj().T
        ),
        "number_of_terms": len(df),
        "support_sizes": str(support_sizes)
    })

    print(name)
    print("Norm:", np.linalg.norm(operator))
    print(
        "Hermiticity error:",
        np.linalg.norm(operator - operator.conj().T)
    )
    print("Number of Pauli strings:", len(df))
    print("Support sizes:", support_sizes)

    display(df)

    print()

cycle_AB_BC_AC
Norm: 16.79654059620611
Hermiticity error: 0.0
Number of Pauli strings: 6
Support sizes: [2]


,motif,pauli_string,coefficient_real,coefficient_imag,support_size
0,cycle_AB_BC_AC,IXXI,-1.656,0.0,2
1,cycle_AB_BC_AC,IYYI,-1.920,0.0,2
2,cycle_AB_BC_AC,IZZI,-1.752,0.0,2
3,cycle_AB_BC_AC,XXII,1.352,0.0,2
4,cycle_AB_BC_AC,YYII,1.996,0.0,2
5,cycle_AB_BC_AC,ZZII,1.524,0.0,2



path_AB_BC_CD
Norm: 12.766172801587796
Hermiticity error: 0.0
Number of Pauli strings: 12
Support sizes: [4]


,motif,pauli_string,coefficient_real,coefficient_imag,support_size
0,path_AB_BC_CD,XYXY,0.875,0.0,4
1,path_AB_BC_CD,XYYX,-0.525,0.0,4
2,path_AB_BC_CD,XZXZ,1.045,0.0,4
3,path_AB_BC_CD,XZZX,-0.825,0.0,4
4,path_AB_BC_CD,YXXY,-0.700,0.0,4
5,path_AB_BC_CD,YXYX,0.420,0.0,4
6,path_AB_BC_CD,YZYZ,0.684,0.0,4
7,path_AB_BC_CD,YZZY,-0.900,0.0,4
8,path_AB_BC_CD,ZXXZ,-1.254,0.0,4
9,path_AB_BC_CD,ZXZX,0.990,0.0,4



star_AB_AC_AD
Norm: 12.205357512174723
Hermiticity error: 0.0
Number of Pauli strings: 12
Support sizes: [4]


,motif,pauli_string,coefficient_real,coefficient_imag,support_size
0,star_AB_AC_AD,XXYY,-0.510,0.0,4
1,star_AB_AC_AD,XXZZ,-0.650,0.0,4
2,star_AB_AC_AD,XYXY,0.816,0.0,4
3,star_AB_AC_AD,XZXZ,0.936,0.0,4
4,star_AB_AC_AD,YXYX,0.690,0.0,4
5,star_AB_AC_AD,YYXX,-1.104,0.0,4
6,star_AB_AC_AD,YYZZ,-0.520,0.0,4
7,star_AB_AC_AD,YZYZ,0.468,0.0,4
8,star_AB_AC_AD,ZXZX,1.150,0.0,4
9,star_AB_AC_AD,ZYZY,0.680,0.0,4



repeated_AB_BC_AB
Norm: 19.281546825916223
Hermiticity error: 0.0
Number of Pauli strings: 6
Support sizes: [2]


,motif,pauli_string,coefficient_real,coefficient_imag,support_size
0,repeated_AB_BC_AB,IXXI,1.872,0.0,2
1,repeated_AB_BC_AB,IYYI,2.684,0.0,2
2,repeated_AB_BC_AB,IZZI,1.148,0.0,2
3,repeated_AB_BC_AB,XIXI,-1.728,0.0,2
4,repeated_AB_BC_AB,YIYI,-2.640,0.0,2
5,repeated_AB_BC_AB,ZIZI,-1.120,0.0,2


In [20]:
cycle_embedding_error = np.linalg.norm(
    M_cycle - np.kron(A_outer_AC, I2)
)

print(
    "Cycle-associator embedding error:",
    cycle_embedding_error
)

Cycle-associator embedding error: 2.0411201962889075e-15


In [21]:
motif_summary = pd.DataFrame(motif_summary)

motif_terms = pd.concat(
    motif_terms,
    ignore_index=True
)

display(motif_summary)

motif_summary.to_csv(
    result_dir / "three_edge_motif_summary.csv",
    index=False
)

motif_terms.to_csv(
    result_dir / "three_edge_motif_pauli_terms.csv",
    index=False
)

,motif,norm,hermiticity_error,number_of_terms,support_sizes
0,cycle_AB_BC_AC,16.796541,0.0,6,[2]
1,path_AB_BC_CD,12.766173,0.0,12,[4]
2,star_AB_AC_AD,12.205358,0.0,12,[4]
3,repeated_AB_BC_AB,19.281547,0.0,6,[2]


edge orderings

In [22]:
# Distinct-edge motifs

ordered_motifs = {
    "cycle": {
        "AB": H4_AB,
        "BC": H4_BC,
        "AC": H4_AC
    },

    "chain": {
        "AB": H4_AB,
        "BC": H4_BC,
        "CD": H4_CD
    },

    "star": {
        "AB": H4_AB,
        "AC": H4_AC,
        "AD": H4_AD
    }
}


ordering_results = []

for motif_name, edge_dict in ordered_motifs.items():

    edge_labels = list(edge_dict.keys())

    for e1, e2, e3 in permutations(edge_labels):

        operator = three_edge_operator(
            edge_dict[e1],
            edge_dict[e2],
            edge_dict[e3]
        )

        operator_norm = np.linalg.norm(operator)

        if operator_norm > tol:

            df = pauli_decomposition(operator, N4)

            support_sizes = sorted(
                int(x)
                for x in df["support_size"].unique()
            )

            number_of_terms = len(df)

        else:

            support_sizes = []
            number_of_terms = 0

        ordering_results.append({
            "motif": motif_name,
            "ordering": f"{e1} → {e2} → {e3}",
            "inner_commutator": f"[{e1},{e2}]",
            "outer_edge": e3,
            "norm": operator_norm,
            "number_of_terms": number_of_terms,
            "support_sizes": str(support_sizes),
            "nonzero": operator_norm > tol
        })


ordering_results = pd.DataFrame(ordering_results)

display(ordering_results)

,motif,ordering,inner_commutator,outer_edge,norm,number_of_terms,support_sizes,nonzero
0,cycle,AB → BC → AC,"[AB,BC]",AC,16.796541,6,[2],True
1,cycle,AB → AC → BC,"[AB,AC]",BC,16.118378,6,[2],True
2,cycle,BC → AB → AC,"[BC,AB]",AC,16.796541,6,[2],True
3,cycle,BC → AC → AB,"[BC,AC]",AB,16.781598,6,[2],True
4,cycle,AC → AB → BC,"[AC,AB]",BC,16.118378,6,[2],True
5,cycle,AC → BC → AB,"[AC,BC]",AB,16.781598,6,[2],True
6,chain,AB → BC → CD,"[AB,BC]",CD,12.766173,12,[4],True
7,chain,AB → CD → BC,"[AB,CD]",BC,0.000000,0,[],False
8,chain,BC → AB → CD,"[BC,AB]",CD,12.766173,12,[4],True
9,chain,BC → CD → AB,"[BC,CD]",AB,12.766173,12,[4],True


In [23]:
ordering_summary = (
    ordering_results
    .groupby("motif")
    .agg(
        total_orderings=("ordering", "count"),
        nonzero_orderings=("nonzero", "sum"),
        min_norm=("norm", "min"),
        max_norm=("norm", "max")
    )
    .reset_index()
)

display(ordering_summary)


ordering_results.to_csv(
    result_dir / "edge_ordering_results.csv",
    index=False
)

ordering_summary.to_csv(
    result_dir / "edge_ordering_summary.csv",
    index=False
)

,motif,total_orderings,nonzero_orderings,min_norm,max_norm
0,chain,6,4,0.000000,12.766173
1,cycle,6,6,16.118378,16.796541
2,star,6,6,12.011744,12.827685


Robustness over random XYZ couplings

In [24]:
# Number of random Hamiltonian samples

n_samples = 100


random_results = []

for sample in tqdm(range(n_samples)):

    # Random XYZ couplings for five edges

    J_AB_r = rng.uniform(0.6, 1.4, size=3)
    J_BC_r = rng.uniform(0.6, 1.4, size=3)
    J_AC_r = rng.uniform(0.6, 1.4, size=3)
    J_CD_r = rng.uniform(0.6, 1.4, size=3)
    J_AD_r = rng.uniform(0.6, 1.4, size=3)

    # Four-qubit edge Hamiltonians

    H_AB_r = edge_hamiltonian(A, B, J_AB_r, N4)
    H_BC_r = edge_hamiltonian(B, C, J_BC_r, N4)
    H_AC_r = edge_hamiltonian(A, C, J_AC_r, N4)
    H_CD_r = edge_hamiltonian(C, D, J_CD_r, N4)
    H_AD_r = edge_hamiltonian(A, D, J_AD_r, N4)

    random_motifs = {
        "cycle": {
            "AB": H_AB_r,
            "BC": H_BC_r,
            "AC": H_AC_r
        },

        "chain": {
            "AB": H_AB_r,
            "BC": H_BC_r,
            "CD": H_CD_r
        },

        "star": {
            "AB": H_AB_r,
            "AC": H_AC_r,
            "AD": H_AD_r
        }
    }

    # Test every edge ordering

    for motif_name, edge_dict in random_motifs.items():

        edge_labels = list(edge_dict.keys())

        for e1, e2, e3 in permutations(edge_labels):

            operator = three_edge_operator(
                edge_dict[e1],
                edge_dict[e2],
                edge_dict[e3]
            )

            operator_norm = np.linalg.norm(operator)

            if operator_norm > tol:

                df = pauli_decomposition(operator, N4)

                support_sizes = sorted(
                    int(x)
                    for x in df["support_size"].unique()
                )

            else:

                support_sizes = []

            random_results.append({
                "sample": sample,
                "motif": motif_name,
                "ordering": f"{e1} → {e2} → {e3}",
                "norm": operator_norm,
                "nonzero": operator_norm > tol,
                "support_sizes": str(support_sizes)
            })


random_results = pd.DataFrame(random_results)

display(random_results.head())

  0%|          | 0/100 [00:00<?, ?it/s]

,sample,motif,ordering,norm,nonzero,support_sizes
0,0,cycle,AB → BC → AC,17.623591,True,[2]
1,0,cycle,AB → AC → BC,17.930760,True,[2]
2,0,cycle,BC → AB → AC,17.623591,True,[2]
3,0,cycle,BC → AC → AB,18.704246,True,[2]
4,0,cycle,AC → AB → BC,17.930760,True,[2]


In [25]:
random_summary = (
    random_results
    .groupby(["motif", "ordering"])
    .agg(
        nonzero_fraction=("nonzero", "mean"),
        mean_norm=("norm", "mean"),
        min_norm=("norm", "min"),
        max_norm=("norm", "max"),
        observed_supports=(
            "support_sizes",
            lambda x: sorted(set(x))
        )
    )
    .reset_index()
)

display(random_summary)

random_results.to_csv(
    result_dir / "random_motif_results.csv",
    index=False
)

random_summary.to_csv(
    result_dir / "random_motif_summary.csv",
    index=False
)

,motif,ordering,nonzero_fraction,mean_norm,min_norm,max_norm,observed_supports
0,chain,AB → BC → CD,1.0,14.561520,6.843437,24.226522,[[4]]
1,chain,AB → CD → BC,0.0,0.000000,0.000000,0.000000,[[]]
2,chain,BC → AB → CD,1.0,14.561520,6.843437,24.226522,[[4]]
3,chain,BC → CD → AB,1.0,14.561520,6.843437,24.226522,[[4]]
4,chain,CD → AB → BC,0.0,0.000000,0.000000,0.000000,[[]]
5,chain,CD → BC → AB,1.0,14.561520,6.843437,24.226522,[[4]]
6,cycle,AB → AC → BC,1.0,20.008458,9.532240,32.923825,[[2]]
7,cycle,AB → BC → AC,1.0,20.100600,9.581531,32.328631,[[2]]
8,cycle,AC → AB → BC,1.0,20.008458,9.532240,32.923825,[[2]]
9,cycle,AC → BC → AB,1.0,20.076961,9.551685,32.864755,[[2]]


In [26]:
motif_robustness = (
    random_results
    .groupby("motif")
    .agg(
        total_cases=("nonzero", "count"),
        nonzero_cases=("nonzero", "sum"),
        nonzero_fraction=("nonzero", "mean"),
        observed_supports=(
            "support_sizes",
            lambda x: sorted(set(x))
        )
    )
    .reset_index()
)

display(motif_robustness)

,motif,total_cases,nonzero_cases,nonzero_fraction,observed_supports
0,chain,600,400,0.666667,"[[4], []]"
1,cycle,600,600,1.000000,[[2]]
2,star,600,600,1.000000,[[4]]


Generic bilinear pair interactions

In [27]:
# General bilinear two-qubit Hamiltonian

def general_edge_hamiltonian(i, j, J, N):
    """
    H_ij = sum_{mu,nu} J[mu,nu] sigma_i^mu sigma_j^nu

    J is a real 3 x 3 matrix.
    """

    pauli_list = [X, Y, Z]

    H = np.zeros((2**N, 2**N), dtype=complex)

    for mu in range(3):
        for nu in range(3):

            H += (
                J[mu, nu]
                * two_body(
                    pauli_list[mu],
                    pauli_list[nu],
                    i, j, N
                )
            )

    return H

In [28]:
n_samples_general = 100

general_results = []


for sample in tqdm(range(n_samples_general)):

    # General real 3 x 3 coupling matrices

    J_AB_g = rng.uniform(-1, 1, size=(3, 3))
    J_BC_g = rng.uniform(-1, 1, size=(3, 3))
    J_AC_g = rng.uniform(-1, 1, size=(3, 3))
    J_CD_g = rng.uniform(-1, 1, size=(3, 3))
    J_AD_g = rng.uniform(-1, 1, size=(3, 3))


    # Edge Hamiltonians

    H_AB_g = general_edge_hamiltonian(
        A, B, J_AB_g, N4
    )

    H_BC_g = general_edge_hamiltonian(
        B, C, J_BC_g, N4
    )

    H_AC_g = general_edge_hamiltonian(
        A, C, J_AC_g, N4
    )

    H_CD_g = general_edge_hamiltonian(
        C, D, J_CD_g, N4
    )

    H_AD_g = general_edge_hamiltonian(
        A, D, J_AD_g, N4
    )


    general_motifs = {
        "cycle": {
            "AB": H_AB_g,
            "BC": H_BC_g,
            "AC": H_AC_g
        },

        "chain": {
            "AB": H_AB_g,
            "BC": H_BC_g,
            "CD": H_CD_g
        },

        "star": {
            "AB": H_AB_g,
            "AC": H_AC_g,
            "AD": H_AD_g
        }
    }


    for motif_name, edge_dict in general_motifs.items():

        edge_labels = list(edge_dict.keys())

        for e1, e2, e3 in permutations(edge_labels):

            operator = three_edge_operator(
                edge_dict[e1],
                edge_dict[e2],
                edge_dict[e3]
            )

            operator_norm = np.linalg.norm(operator)

            if operator_norm > tol:

                df = pauli_decomposition(operator, N4)

                support_sizes = sorted(
                    int(x)
                    for x in df["support_size"].unique()
                )

                number_of_terms = len(df)

            else:

                support_sizes = []
                number_of_terms = 0


            general_results.append({
                "sample": sample,
                "motif": motif_name,
                "ordering": f"{e1} → {e2} → {e3}",
                "norm": operator_norm,
                "number_of_terms": number_of_terms,
                "support_sizes": str(support_sizes),
                "nonzero": operator_norm > tol
            })


general_results = pd.DataFrame(general_results)

  0%|          | 0/100 [00:00<?, ?it/s]

In [29]:
general_summary = (
    general_results
    .groupby("motif")
    .agg(
        total_cases=("nonzero", "count"),
        nonzero_cases=("nonzero", "sum"),
        nonzero_fraction=("nonzero", "mean"),
        observed_supports=(
            "support_sizes",
            lambda x: sorted(set(x))
        ),
        observed_term_counts=(
            "number_of_terms",
            lambda x: sorted(set(x))
        )
    )
    .reset_index()
)

display(general_summary)


general_results.to_csv(
    result_dir / "general_bilinear_motif_results.csv",
    index=False
)

general_summary.to_csv(
    result_dir / "general_bilinear_motif_summary.csv",
    index=False
)

,motif,total_cases,nonzero_cases,nonzero_fraction,observed_supports,observed_term_counts
0,chain,600,400,0.666667,"[[4], []]","[0, 81]"
1,cycle,600,600,1.000000,[[2]],[18]
2,star,600,600,1.000000,[[4]],[81]


Exhaustive catalogue of all three-edge sequences

In [31]:
# All six edges on four nodes

node_names = {
    A: "A",
    B: "B",
    C: "C",
    D: "D"
}

all_edges = list(combinations(range(N4), 2))


def edge_name(edge):
    i, j = edge
    return node_names[i] + node_names[j]


# Classify the topology of three edges

def classify_three_edges(edges):

    distinct_edges = list(set(edges))
    n_distinct = len(distinct_edges)

    if n_distinct == 1:
        return "single_edge_repeated"

    if n_distinct == 2:

        e1, e2 = distinct_edges
        overlap = len(set(e1) & set(e2))

        if overlap == 1:
            return "two_adjacent_edges"

        else:
            return "two_disjoint_edges"

    # Three distinct edges

    degrees = {}

    for i, j in distinct_edges:
        degrees[i] = degrees.get(i, 0) + 1
        degrees[j] = degrees.get(j, 0) + 1

    degree_sequence = sorted(
        degrees.values(),
        reverse=True
    )

    if degree_sequence == [2, 2, 2]:
        return "cycle"

    if degree_sequence == [3, 1, 1, 1]:
        return "star"

    if degree_sequence == [2, 2, 1, 1]:
        return "chain"

    return "other"

In [32]:
# Independent random couplings for this catalogue

rng_catalogue = np.random.default_rng(seed + 10)


H_edges = {}

for edge in all_edges:

    J = rng_catalogue.uniform(
        -1, 1,
        size=(3, 3)
    )

    H_edges[edge] = general_edge_hamiltonian(
        edge[0],
        edge[1],
        J,
        N4
    )

In [33]:
catalogue_results = []


for e1, e2, e3 in product(
    all_edges,
    repeat=3
):

    operator = three_edge_operator(
        H_edges[e1],
        H_edges[e2],
        H_edges[e3]
    )

    operator_norm = np.linalg.norm(operator)

    if operator_norm > tol:

        df = pauli_decomposition(
            operator,
            N4
        )

        support_sizes = sorted(
            int(x)
            for x in df["support_size"].unique()
        )

    else:

        support_sizes = []


    catalogue_results.append({

        "motif": classify_three_edges(
            [e1, e2, e3]
        ),

        "ordering": (
            f"{edge_name(e1)} → "
            f"{edge_name(e2)} → "
            f"{edge_name(e3)}"
        ),

        "inner_edges": (
            f"{edge_name(e1)}, "
            f"{edge_name(e2)}"
        ),

        "n_distinct_edges": len(
            set([e1, e2, e3])
        ),

        "n_union_nodes": len(
            set(e1) | set(e2) | set(e3)
        ),

        "norm": operator_norm,

        "nonzero": operator_norm > tol,

        "support_sizes": str(
            support_sizes
        )
    })


catalogue_results = pd.DataFrame(
    catalogue_results
)

display(catalogue_results.head())

,motif,ordering,inner_edges,n_distinct_edges,n_union_nodes,norm,nonzero,support_sizes
0,single_edge_repeated,AB → AB → AB,"AB, AB",1,2,0.0,False,[]
1,two_adjacent_edges,AB → AB → AC,"AB, AB",2,3,0.0,False,[]
2,two_adjacent_edges,AB → AB → AD,"AB, AB",2,3,0.0,False,[]
3,two_adjacent_edges,AB → AB → BC,"AB, AB",2,3,0.0,False,[]
4,two_adjacent_edges,AB → AB → BD,"AB, AB",2,3,0.0,False,[]


In [34]:
catalogue_summary = (
    catalogue_results
    .groupby("motif")
    .agg(
        total_orderings=("ordering", "count"),
        nonzero_orderings=("nonzero", "sum"),
        nonzero_fraction=("nonzero", "mean"),
        observed_supports=(
            "support_sizes",
            lambda x: sorted(set(x))
        )
    )
    .reset_index()
)

display(catalogue_summary)


catalogue_results.to_csv(
    result_dir / "complete_three_edge_catalogue.csv",
    index=False
)

catalogue_summary.to_csv(
    result_dir / "complete_three_edge_summary.csv",
    index=False
)

,motif,total_orderings,nonzero_orderings,nonzero_fraction,observed_supports
0,chain,72,48,0.666667,"[[4], []]"
1,cycle,24,24,1.000000,[[2]]
2,single_edge_repeated,6,0,0.000000,[[]]
3,star,24,24,1.000000,[[4]]
4,two_adjacent_edges,72,48,0.666667,"[[2], []]"
5,two_disjoint_edges,18,0,0.000000,[[]]


单串选择定则与 Kitaev 熄灭

In [35]:
# Single Pauli-string edge operator

def single_edge_word(i, j, letter_i, letter_j, N=3):

    return two_body(
        pauli_basis[letter_i],
        pauli_basis[letter_j],
        i, j, N
    )


single_word_results = []


for a, b, c, d, e, f in product(
    "XYZ",
    repeat=6
):

    # Edge words:
    # AB = a_A b_B
    # BC = c_B d_C
    # AC = e_A f_C

    P_AB = single_edge_word(
        A, B, a, b
    )

    P_BC = single_edge_word(
        B, C, c, d
    )

    P_AC = single_edge_word(
        A, C, e, f
    )


    # Clash bits at the three vertices

    beta_A = int(a != e)
    beta_B = int(b != c)
    beta_C = int(d != f)

    clash_total = (
        beta_A
        + beta_B
        + beta_C
    )


    # Three triangle channels

    C_outer_AB = 0.25 * commutator(
        P_AB,
        commutator(P_AC, P_BC)
    )

    C_outer_BC = 0.25 * commutator(
        P_BC,
        commutator(P_AB, P_AC)
    )

    C_outer_AC = 0.25 * commutator(
        P_AC,
        commutator(P_AB, P_BC)
    )


    norm_AB = np.linalg.norm(C_outer_AB)
    norm_BC = np.linalg.norm(C_outer_BC)
    norm_AC = np.linalg.norm(C_outer_AC)

    n_nonzero = sum([
        norm_AB > tol,
        norm_BC > tol,
        norm_AC > tol
    ])


    single_word_results.append({

        "P_AB": a + b,
        "P_BC": c + d,
        "P_AC": e + f,

        "clash_A": beta_A,
        "clash_B": beta_B,
        "clash_C": beta_C,

        "clash_pattern": (
            f"{beta_A}{beta_B}{beta_C}"
        ),

        "clash_total": clash_total,

        "norm_outer_AB": norm_AB,
        "norm_outer_BC": norm_BC,
        "norm_outer_AC": norm_AC,

        "n_nonzero_channels": n_nonzero
    })


single_word_results = pd.DataFrame(
    single_word_results
)

In [36]:
selection_summary = (
    single_word_results
    .groupby([
        "clash_total",
        "n_nonzero_channels"
    ])
    .size()
    .reset_index(
        name="number_of_word_triples"
    )
)

display(selection_summary)

,clash_total,n_nonzero_channels,number_of_word_triples
0,0,0,27
1,1,0,162
2,2,2,324
3,3,0,216


In [37]:
pattern_summary = (
    single_word_results
    .groupby("clash_pattern")
    .agg(
        number_of_word_triples=(
            "P_AB",
            "count"
        ),

        clash_total=(
            "clash_total",
            "first"
        ),

        nonzero_channels=(
            "n_nonzero_channels",
            lambda x: sorted(set(x))
        )
    )
    .reset_index()
)

display(pattern_summary)


single_word_results.to_csv(
    result_dir / "single_word_selection_rule.csv",
    index=False
)

selection_summary.to_csv(
    result_dir / "single_word_selection_summary.csv",
    index=False
)

,clash_pattern,number_of_word_triples,clash_total,nonzero_channels
0,000,27,0,[0]
1,001,54,1,[0]
2,010,54,1,[0]
3,011,108,2,[2]
4,100,54,1,[0]
5,101,108,2,[2]
6,110,108,2,[2]
7,111,216,3,[0]


In [38]:
# Kitaev-type all-conflict example

K_AB = single_edge_word(
    A, B, "X", "X"
)

K_BC = single_edge_word(
    B, C, "Y", "Y"
)

K_AC = single_edge_word(
    A, C, "Z", "Z"
)


# Ordinary pairwise commutators

kitaev_pairwise_norms = [
    np.linalg.norm(
        commutator(K_AB, K_BC)
    ),

    np.linalg.norm(
        commutator(K_AB, K_AC)
    ),

    np.linalg.norm(
        commutator(K_BC, K_AC)
    )
]


# Triangle nested channels

kitaev_nested_norms = [
    np.linalg.norm(
        0.25 * commutator(
            K_AB,
            commutator(K_AC, K_BC)
        )
    ),

    np.linalg.norm(
        0.25 * commutator(
            K_BC,
            commutator(K_AB, K_AC)
        )
    ),

    np.linalg.norm(
        0.25 * commutator(
            K_AC,
            commutator(K_AB, K_BC)
        )
    )
]


print("Kitaev-type pairwise commutator norms:")
print(kitaev_pairwise_norms)

print()
print("Kitaev-type nested-channel norms:")
print(kitaev_nested_norms)

Kitaev-type pairwise commutator norms:
[np.float64(5.656854249492381), np.float64(5.656854249492381), np.float64(5.656854249492381)]

Kitaev-type nested-channel norms:
[np.float64(0.0), np.float64(0.0), np.float64(0.0)]


Jacobi identity and independent cycle channels

In [40]:
# Generic triangle Hamiltonian for the Jacobi test

rng_jacobi = np.random.default_rng(seed + 20)

J_AB_j = rng_jacobi.uniform(-1, 1, size=(3, 3))
J_BC_j = rng_jacobi.uniform(-1, 1, size=(3, 3))
J_AC_j = rng_jacobi.uniform(-1, 1, size=(3, 3))

H_AB_j = general_edge_hamiltonian(
    A, B, J_AB_j, N
)

H_BC_j = general_edge_hamiltonian(
    B, C, J_BC_j, N
)

H_AC_j = general_edge_hamiltonian(
    A, C, J_AC_j, N
)

In [41]:
# Three outer-edge channels

C_AB = 0.25 * commutator(
    H_AB_j,
    commutator(H_AC_j, H_BC_j)
)

C_BC = 0.25 * commutator(
    H_BC_j,
    commutator(H_AB_j, H_AC_j)
)

C_AC = 0.25 * commutator(
    H_AC_j,
    commutator(H_AB_j, H_BC_j)
)

In [42]:
# Jacobi identity

jacobi_residual = (
    C_AC
    - C_AB
    - C_BC
)

print("Channel norms:")
print("||C_AB|| =", np.linalg.norm(C_AB))
print("||C_BC|| =", np.linalg.norm(C_BC))
print("||C_AC|| =", np.linalg.norm(C_AC))

print()
print(
    "Jacobi residual ||C_AC - C_AB - C_BC|| =",
    np.linalg.norm(jacobi_residual)
)

Channel norms:
||C_AB|| = 13.020092437610213
||C_BC|| = 12.124409292844225
||C_AC|| = 8.067948776162845

Jacobi residual ||C_AC - C_AB - C_BC|| = 1.9062645721593414e-15


In [43]:
# Linear independence of the three channels

channel_matrix = np.column_stack([
    C_AB.reshape(-1),
    C_BC.reshape(-1),
    C_AC.reshape(-1)
])

singular_values = np.linalg.svd(
    channel_matrix,
    compute_uv=False
)

channel_rank = np.linalg.matrix_rank(
    channel_matrix,
    tol=tol
)

print()
print("Singular values:")
print(singular_values)

print()
print("Number of independent cycle channels:")
print(channel_rank)


Singular values:
[16.91147  9.77845  0.     ]

Number of independent cycle channels:
2


In [44]:
# All six orderings on the triangle

six_orderings = {
    "AB → BC → AC":
        three_edge_operator(
            H_AB_j, H_BC_j, H_AC_j
        ),

    "BC → AB → AC":
        three_edge_operator(
            H_BC_j, H_AB_j, H_AC_j
        ),

    "AC → BC → AB":
        three_edge_operator(
            H_AC_j, H_BC_j, H_AB_j
        ),

    "BC → AC → AB":
        three_edge_operator(
            H_BC_j, H_AC_j, H_AB_j
        ),

    "AB → AC → BC":
        three_edge_operator(
            H_AB_j, H_AC_j, H_BC_j
        ),

    "AC → AB → BC":
        three_edge_operator(
            H_AC_j, H_AB_j, H_BC_j
        )
}


ordering_checks = pd.DataFrame([
    {
        "relation": (
            "(AB → BC → AC) "
            "+ (BC → AB → AC)"
        ),
        "error": np.linalg.norm(
            six_orderings["AB → BC → AC"]
            + six_orderings["BC → AB → AC"]
        )
    },

    {
        "relation": (
            "(AC → BC → AB) "
            "+ (BC → AC → AB)"
        ),
        "error": np.linalg.norm(
            six_orderings["AC → BC → AB"]
            + six_orderings["BC → AC → AB"]
        )
    },

    {
        "relation": (
            "(AB → AC → BC) "
            "+ (AC → AB → BC)"
        ),
        "error": np.linalg.norm(
            six_orderings["AB → AC → BC"]
            + six_orderings["AC → AB → BC"]
        )
    },

    {
        "relation": "C_AC - C_AB - C_BC",
        "error": np.linalg.norm(
            C_AC - C_AB - C_BC
        )
    }
])

display(ordering_checks)

ordering_checks.to_csv(
    result_dir / "jacobi_ordering_checks.csv",
    index=False
)

,relation,error
0,(AB → BC → AC) + (BC → AB → AC),0.000000e+00
1,(AC → BC → AB) + (BC → AC → AB),0.000000e+00
2,(AB → AC → BC) + (AC → AB → BC),0.000000e+00
3,C_AC - C_AB - C_BC,1.906265e-15


缺口边生成的 iff 条件与方向性

In [45]:
# Return the third Pauli letter

def third_pauli(letter_1, letter_2):

    if letter_1 == letter_2:
        return None

    return list(
        set("XYZ") - {letter_1, letter_2}
    )[0]


# Check whether an operator has support exactly on A and C

def is_gap_edge(operator, tol=1e-10):

    if np.linalg.norm(operator) < tol:
        return False

    df = pauli_decomposition(
        operator,
        N=3,
        tol=tol
    )

    # For a single-word calculation, the result should contain
    # either zero or one Pauli string.

    for word in df["pauli_string"]:

        active_sites = {
            i for i, letter in enumerate(word)
            if letter != "I"
        }

        if active_sites != {A, C}:
            return False

    return len(df) > 0

In [46]:
gap_21_results = []


for a, b, a2, b2, c, d in product(
    "XYZ",
    repeat=6
):

    # P_AB  = a_A b_B
    # P2_AB = a2_A b2_B
    # Q_BC  = c_B d_C

    P_AB = single_edge_word(
        A, B, a, b
    )

    P2_AB = single_edge_word(
        A, B, a2, b2
    )

    Q_BC = single_edge_word(
        B, C, c, d
    )


    # (2,1) nested commutator

    operator = 0.25 * commutator(
        P2_AB,
        commutator(P_AB, Q_BC)
    )


    # Numerical result

    observed_gap = is_gap_edge(operator)


    # Predicted iff conditions

    inner_conflict_B = (b != c)

    inner_B_letter = third_pauli(b, c)

    cancel_B = (
        inner_conflict_B
        and b2 == inner_B_letter
    )

    preserve_A = (a2 != a)


    predicted_gap = (
        inner_conflict_B
        and cancel_B
        and preserve_A
    )


    # Output Pauli word, when nonzero

    if np.linalg.norm(operator) > tol:

        df = pauli_decomposition(
            operator,
            N=3,
            tol=tol
        )

        output_words = ",".join(
            df["pauli_string"]
        )

    else:

        output_words = ""


    gap_21_results.append({

        "P_AB": a + b,
        "P2_AB": a2 + b2,
        "Q_BC": c + d,

        "inner_conflict_B": inner_conflict_B,
        "cancel_B": cancel_B,
        "preserve_A": preserve_A,

        "predicted_gap": predicted_gap,
        "observed_gap": observed_gap,

        "operator_norm": np.linalg.norm(operator),
        "output_words": output_words
    })


gap_21_results = pd.DataFrame(
    gap_21_results
)

In [47]:
gap_21_check = pd.crosstab(
    gap_21_results["predicted_gap"],
    gap_21_results["observed_gap"]
)

display(gap_21_check)


n_mismatches_21 = np.sum(
    gap_21_results["predicted_gap"]
    != gap_21_results["observed_gap"]
)

print(
    "Number of iff mismatches for (2,1):",
    n_mismatches_21
)

print(
    "Number of generated AC gap edges:",
    gap_21_results["observed_gap"].sum()
)

observed_gap,False,True
predicted_gap,,
False,621,0
True,0,108


Number of iff mismatches for (2,1): 0
Number of generated AC gap edges: 108


In [48]:
gap_12_results = []


for a, b, c, d, c2, d2 in product(
    "XYZ",
    repeat=6
):

    # P_AB  = a_A b_B
    # Q_BC  = c_B d_C
    # Q2_BC = c2_B d2_C

    P_AB = single_edge_word(
        A, B, a, b
    )

    Q_BC = single_edge_word(
        B, C, c, d
    )

    Q2_BC = single_edge_word(
        B, C, c2, d2
    )


    # (1,2) nested commutator

    operator = 0.25 * commutator(
        Q2_BC,
        commutator(Q_BC, P_AB)
    )


    # Numerical result

    observed_gap = is_gap_edge(operator)


    # Predicted mirror conditions

    inner_conflict_B = (c != b)

    inner_B_letter = third_pauli(c, b)

    cancel_B = (
        inner_conflict_B
        and c2 == inner_B_letter
    )

    preserve_C = (d2 != d)


    predicted_gap = (
        inner_conflict_B
        and cancel_B
        and preserve_C
    )


    if np.linalg.norm(operator) > tol:

        df = pauli_decomposition(
            operator,
            N=3,
            tol=tol
        )

        output_words = ",".join(
            df["pauli_string"]
        )

    else:

        output_words = ""


    gap_12_results.append({

        "P_AB": a + b,
        "Q_BC": c + d,
        "Q2_BC": c2 + d2,

        "inner_conflict_B": inner_conflict_B,
        "cancel_B": cancel_B,
        "preserve_C": preserve_C,

        "predicted_gap": predicted_gap,
        "observed_gap": observed_gap,

        "operator_norm": np.linalg.norm(operator),
        "output_words": output_words
    })


gap_12_results = pd.DataFrame(
    gap_12_results
)

In [49]:
gap_12_check = pd.crosstab(
    gap_12_results["predicted_gap"],
    gap_12_results["observed_gap"]
)

display(gap_12_check)


n_mismatches_12 = np.sum(
    gap_12_results["predicted_gap"]
    != gap_12_results["observed_gap"]
)

print(
    "Number of iff mismatches for (1,2):",
    n_mismatches_12
)

print(
    "Number of generated AC gap edges:",
    gap_12_results["observed_gap"].sum()
)

observed_gap,False,True
predicted_gap,,
False,621,0
True,0,108


Number of iff mismatches for (1,2): 0
Number of generated AC gap edges: 108


In [50]:
gap_21_results.to_csv(
    result_dir / "gap_edge_21_iff.csv",
    index=False
)

gap_12_results.to_csv(
    result_dir / "gap_edge_12_iff.csv",
    index=False
)

全冲突闭环的词级熄灭

In [51]:
# All binary bracketings of an ordered list

def all_bracketings(labels):

    if len(labels) == 1:
        return [labels[0]]

    trees = []

    for split in range(1, len(labels)):

        left_labels = labels[:split]
        right_labels = labels[split:]

        for left in all_bracketings(left_labels):
            for right in all_bracketings(right_labels):

                trees.append(
                    (left, right)
                )

    return trees


# Evaluate a nested-commutator tree

def evaluate_commutator_tree(tree, operators):

    if isinstance(tree, str):
        return operators[tree]

    left, right = tree

    return commutator(
        evaluate_commutator_tree(
            left,
            operators
        ),
        evaluate_commutator_tree(
            right,
            operators
        )
    )


# Convert tree to readable notation

def tree_to_string(tree):

    if isinstance(tree, str):
        return tree

    left, right = tree

    return (
        f"[{tree_to_string(left)},"
        f"{tree_to_string(right)}]"
    )

In [52]:
def enumerate_multilinear_words(
    operators,
    system_name,
    case_name,
    N
):

    results = []

    edge_labels = list(
        operators.keys()
    )

    for ordering in permutations(edge_labels):

        for tree in all_bracketings(
            list(ordering)
        ):

            operator = evaluate_commutator_tree(
                tree,
                operators
            )

            operator_norm = np.linalg.norm(
                operator
            )

            if operator_norm > tol:

                df = pauli_decomposition(
                    operator,
                    N
                )

                support_sizes = sorted(
                    int(x)
                    for x in df["support_size"].unique()
                )

                output_words = ",".join(
                    df["pauli_string"]
                )

            else:

                support_sizes = []
                output_words = ""

            results.append({

                "system": system_name,
                "case": case_name,

                "ordering": " → ".join(
                    ordering
                ),

                "bracketing": tree_to_string(
                    tree
                ),

                "norm": operator_norm,
                "nonzero": operator_norm > tol,

                "support_sizes": str(
                    support_sizes
                ),

                "output_words": output_words
            })

    return pd.DataFrame(results)

In [53]:
triangle_all_conflict = {

    "AB": single_edge_word(
        A, B, "X", "X", N=3
    ),

    "BC": single_edge_word(
        B, C, "Y", "Y", N=3
    ),

    "AC": single_edge_word(
        A, C, "Z", "Z", N=3
    )
}


triangle_control = {

    "AB": single_edge_word(
        A, B, "X", "X", N=3
    ),

    "BC": single_edge_word(
        B, C, "Y", "Y", N=3
    ),

    # A is now matching: X on AB and X on AC
    "AC": single_edge_word(
        A, C, "X", "Z", N=3
    )
}

In [54]:
square_all_conflict = {

    "AB": single_edge_word(
        A, B, "X", "X", N=4
    ),

    "BC": single_edge_word(
        B, C, "Y", "Y", N=4
    ),

    "CD": single_edge_word(
        C, D, "X", "X", N=4
    ),

    "DA": single_edge_word(
        D, A, "Y", "Y", N=4
    )
}


square_control = {

    "AB": single_edge_word(
        A, B, "X", "X", N=4
    ),

    "BC": single_edge_word(
        B, C, "Y", "Y", N=4
    ),

    "CD": single_edge_word(
        C, D, "X", "X", N=4
    ),

    # D is now matching: X on CD and X on DA
    "DA": single_edge_word(
        D, A, "X", "Y", N=4
    )
}

In [55]:
triangle_conflict_results = enumerate_multilinear_words(
    triangle_all_conflict,
    system_name="triangle",
    case_name="all_conflict",
    N=3
)

triangle_control_results = enumerate_multilinear_words(
    triangle_control,
    system_name="triangle",
    case_name="one_matching_vertex",
    N=3
)

square_conflict_results = enumerate_multilinear_words(
    square_all_conflict,
    system_name="square",
    case_name="all_conflict",
    N=4
)

square_control_results = enumerate_multilinear_words(
    square_control,
    system_name="square",
    case_name="one_matching_vertex",
    N=4
)


full_conflict_results = pd.concat(
    [
        triangle_conflict_results,
        triangle_control_results,
        square_conflict_results,
        square_control_results
    ],
    ignore_index=True
)

In [56]:
full_conflict_summary = (
    full_conflict_results
    .groupby([
        "system",
        "case"
    ])
    .agg(
        total_words=("nonzero", "count"),
        nonzero_words=("nonzero", "sum"),
        max_norm=("norm", "max"),
        observed_supports=(
            "support_sizes",
            lambda x: sorted(set(x))
        ),
        observed_outputs=(
            "output_words",
            lambda x: sorted(
                value
                for value in set(x)
                if value != ""
            )
        )
    )
    .reset_index()
)

display(full_conflict_summary)


full_conflict_results.to_csv(
    result_dir / "full_conflict_word_results.csv",
    index=False
)

full_conflict_summary.to_csv(
    result_dir / "full_conflict_word_summary.csv",
    index=False
)

,system,case,total_words,nonzero_words,max_norm,observed_supports,observed_outputs
0,square,all_conflict,120,0,0.000000,[[]],[]
1,square,one_matching_vertex,120,40,32.000000,"[[3], []]",[ZZZI]
2,triangle,all_conflict,12,0,0.000000,[[]],[]
3,triangle,one_matching_vertex,12,8,11.313708,"[[2], []]",[IZX]


BCH protocol selects a cycle direction

In [69]:
from scipy.linalg import logm


# Protocol edges

H1 = H_AB_j
H2 = H_BC_j
H3 = H_AC_j


def protocol_log(H1, H2, H3, s1, s2, s3):
    """
    Chronological pulse order:
        H1 -> H2 -> H3

    Since operators act on states from right to left:
        U = U3 @ U2 @ U1
          = exp(-i s3 H3)
            exp(-i s2 H2)
            exp(-i s1 H1)
    """

    U = (
        expm(-1j * s3 * H3)
        @ expm(-1j * s2 * H2)
        @ expm(-1j * s1 * H1)
    )

    Omega = logm(U)

    Omega = 0.5 * (
        Omega - Omega.conj().T
    )

    return Omega

In [70]:
def protocol_log_2(H1, H2, s1, s2):
    """
    Chronological pulse order:
        H1 -> H2

    Operators act on states from right to left:
        U = exp(-i s2 H2) exp(-i s1 H1)
    """

    U = (
        expm(-1j * s2 * H2)
        @ expm(-1j * s1 * H1)
    )

    L = logm(U)

    # Keep the anti-Hermitian part
    L = 0.5 * (
        L - L.conj().T
    )

    return L

In [71]:
# Second-order BCH sign anchor
#
# Chronological convention:
#     H1 -> H2
#
# Therefore:
#     U = exp(-i s2 H2) exp(-i s1 H1)
#
# The s1*s2 coefficient of log U must be:
#     +1/2 [H1, H2]

h = 0.004

L_11 = sum(
    sign1 * sign2
    * protocol_log_2(
        H1,
        H2,
        sign1 * h,
        sign2 * h
    )
    for sign1, sign2 in product(
        [-1, 1],
        repeat=2
    )
) / (4 * h**2)


L_11_target = (
    0.5
    * commutator(H1, H2)
)


relative_error_11 = (
    np.linalg.norm(
        L_11 - L_11_target
    )
    / np.linalg.norm(L_11_target)
)


print(
    "Second-order sign-anchor error:",
    relative_error_11
)


assert relative_error_11 < 1e-3

Second-order sign-anchor error: 7.091421272701631e-11


In [72]:
# Reverse chronological order:
#     H2 -> H1
#
# The same degree-2 component must flip sign.

L_11_reversed = sum(
    sign1 * sign2
    * protocol_log_2(
        H2,
        H1,
        sign2 * h,
        sign1 * h
    )
    for sign1, sign2 in product(
        [-1, 1],
        repeat=2
    )
) / (4 * h**2)


reverse_target = (
    -0.5
    * commutator(H1, H2)
)


reverse_relative_error = (
    np.linalg.norm(
        L_11_reversed - reverse_target
    )
    / np.linalg.norm(reverse_target)
)


print(
    "Reversed-order sign-anchor error:",
    reverse_relative_error
)


assert reverse_relative_error < 1e-3

Reversed-order sign-anchor error: 7.088316539903581e-11


In [58]:
def extract_cycle_coefficient(H1, H2, H3, h):
    """
    Extract the coefficient of s1*s2*s3 in log U
    using an eight-point central difference.
    """

    Omega_111 = np.zeros_like(
        H1,
        dtype=complex
    )

    for sign1, sign2, sign3 in product(
        [-1, 1],
        repeat=3
    ):

        Omega_111 += (
            sign1 * sign2 * sign3
            * protocol_log(
                H1,
                H2,
                H3,
                sign1 * h,
                sign2 * h,
                sign3 * h
            )
        )

    Omega_111 /= 8 * h**3

    return Omega_111

In [59]:
# Analytic BCH multilinear coefficient
# for the chronological protocol H1 -> H2 -> H3.
#
# Important:
# the nesting order inside the commutators is not
# the chronological pulse order.


Omega_111_analytic = 1j * (
    (1 / 3)
    * commutator(
        H1,
        commutator(H2, H3)
    )

    - (1 / 6)
    * commutator(
        H2,
        commutator(H1, H3)
    )
)


print(
    "Anti-Hermiticity error:",
    np.linalg.norm(
        Omega_111_analytic
        + Omega_111_analytic.conj().T
    )
)

Anti-Hermiticity error: 1.4517165496329299e-15


In [60]:
# Hermitian effective-generator coefficient

K_111_analytic = 1j * Omega_111_analytic


# Representation in the two-channel associator basis

K_111_from_basis = (
    (4 / 3) * C_AB
    + (2 / 3) * C_BC
)


print(
    "BCH-to-associator basis error:",
    np.linalg.norm(
        K_111_analytic
        - K_111_from_basis
    )
)

BCH-to-associator basis error: 0.0


In [61]:
h_values = [
    0.04,
    0.02,
    0.01,
    0.005,
    0.002
]

bch_results = []


for h in h_values:

    Omega_111_numeric = extract_cycle_coefficient(
        H1,
        H2,
        H3,
        h
    )

    K_111_numeric = 1j * Omega_111_numeric

    absolute_error = np.linalg.norm(
        K_111_numeric
        - K_111_analytic
    )

    relative_error = (
        absolute_error
        / np.linalg.norm(K_111_analytic)
    )

    bch_results.append({
        "h": h,
        "absolute_error": absolute_error,
        "relative_error": relative_error
    })


bch_results = pd.DataFrame(
    bch_results
)

display(bch_results)

,h,absolute_error,relative_error
0,0.040,0.008195,0.000685
1,0.020,0.002053,0.000172
2,0.010,0.000514,0.000043
3,0.005,0.000128,0.000011
4,0.002,0.000021,0.000002


In [62]:
h_projection = h_values[-1]

Omega_111_numeric = extract_cycle_coefficient(
    H1,
    H2,
    H3,
    h_projection
)

K_111_numeric = 1j * Omega_111_numeric


# Basis matrix: columns are C_AB and C_BC

cycle_basis = np.column_stack([
    C_AB.reshape(-1),
    C_BC.reshape(-1)
])


coefficients, _, _, _ = np.linalg.lstsq(
    cycle_basis,
    K_111_numeric.reshape(-1),
    rcond=None
)


direction_result = pd.DataFrame({
    "basis_channel": [
        "C_AB",
        "C_BC"
    ],

    "expected_coefficient": [
        4 / 3,
        2 / 3
    ],

    "numerical_coefficient": [
        coefficients[0].real,
        coefficients[1].real
    ]
})


display(direction_result)


bch_results.to_csv(
    result_dir / "bch_cycle_convergence.csv",
    index=False
)

direction_result.to_csv(
    result_dir / "bch_cycle_direction.csv",
    index=False
)

,basis_channel,expected_coefficient,numerical_coefficient
0,C_AB,1.333333,1.333333
1,C_BC,0.666667,0.666666


Six protocol orderings in the cycle sector

In [64]:
# Fixed physical edge operators

protocol_edges = {
    "AB": H_AB_j,
    "BC": H_BC_j,
    "AC": H_AC_j
}


# Fixed two-dimensional cycle basis

cycle_basis = np.column_stack([
    C_AB.reshape(-1),
    C_BC.reshape(-1)
])


h = 0.002

protocol_direction_results = []


for ordering in permutations(
    protocol_edges.keys()
):

    e1, e2, e3 = ordering

    # e1 acts first, then e2, then e3

    Omega_111 = extract_cycle_coefficient(
        protocol_edges[e1],
        protocol_edges[e2],
        protocol_edges[e3],
        h
    )

    K_111 = 1j * Omega_111


    # Project onto C_AB and C_BC

    coefficients, _, _, _ = np.linalg.lstsq(
        cycle_basis,
        K_111.reshape(-1),
        rcond=None
    )

    reconstructed = (
        coefficients[0] * C_AB
        + coefficients[1] * C_BC
    )

    projection_error = np.linalg.norm(
        K_111 - reconstructed
    )


    protocol_direction_results.append({

        "protocol": (
            f"{e1} → {e2} → {e3}"
        ),

        "first_edge": e1,
        "second_edge": e2,
        "third_edge": e3,

        "coefficient_C_AB":
            coefficients[0].real,

        "coefficient_C_BC":
            coefficients[1].real,

        "generator_norm":
            np.linalg.norm(K_111),

        "projection_error":
            projection_error
    })


protocol_direction_results = pd.DataFrame(
    protocol_direction_results
)

display(protocol_direction_results)

,protocol,first_edge,second_edge,third_edge,coefficient_C_AB,coefficient_C_BC,generator_norm,projection_error
0,AB → BC → AC,AB,BC,AC,1.333333,0.666666,11.967145,0.000016
1,AB → AC → BC,AB,AC,BC,-0.666667,0.666666,15.887878,0.000016
2,BC → AB → AC,BC,AB,AC,-0.666667,-1.333334,10.638795,0.000016
3,BC → AC → AB,BC,AC,AB,-0.666667,0.666666,15.887878,0.000016
4,AC → AB → BC,AC,AB,BC,-0.666667,-1.333334,10.638795,0.000016
5,AC → BC → AB,AC,BC,AB,1.333333,0.666666,11.967145,0.000016


In [65]:
protocol_direction_results[
    "scaled_C_AB"
] = (
    1.5
    * protocol_direction_results[
        "coefficient_C_AB"
    ]
)

protocol_direction_results[
    "scaled_C_BC"
] = (
    1.5
    * protocol_direction_results[
        "coefficient_C_BC"
    ]
)


display(
    protocol_direction_results[
        [
            "protocol",
            "coefficient_C_AB",
            "coefficient_C_BC",
            "scaled_C_AB",
            "scaled_C_BC",
            "projection_error"
        ]
    ]
)

,protocol,coefficient_C_AB,coefficient_C_BC,scaled_C_AB,scaled_C_BC,projection_error
0,AB → BC → AC,1.333333,0.666666,2.0,0.999998,0.000016
1,AB → AC → BC,-0.666667,0.666666,-1.0,0.999998,0.000016
2,BC → AB → AC,-0.666667,-1.333334,-1.0,-2.000002,0.000016
3,BC → AC → AB,-0.666667,0.666666,-1.0,0.999998,0.000016
4,AC → AB → BC,-0.666667,-1.333334,-1.0,-2.000002,0.000016
5,AC → BC → AB,1.333333,0.666666,2.0,0.999998,0.000016


In [66]:
reverse_pairs = [
    (
        "AB → BC → AC",
        "AC → BC → AB"
    ),

    (
        "AB → AC → BC",
        "BC → AC → AB"
    ),

    (
        "BC → AB → AC",
        "AC → AB → BC"
    )
]


reverse_checks = []


for protocol_1, protocol_2 in reverse_pairs:

    row_1 = protocol_direction_results[
        protocol_direction_results[
            "protocol"
        ] == protocol_1
    ].iloc[0]

    row_2 = protocol_direction_results[
        protocol_direction_results[
            "protocol"
        ] == protocol_2
    ].iloc[0]


    direction_1 = np.array([
        row_1["coefficient_C_AB"],
        row_1["coefficient_C_BC"]
    ])

    direction_2 = np.array([
        row_2["coefficient_C_AB"],
        row_2["coefficient_C_BC"]
    ])


    reverse_checks.append({

        "protocol_1": protocol_1,
        "protocol_2": protocol_2,

        "direction_difference":
            np.linalg.norm(
                direction_1
                - direction_2
            )
    })


reverse_checks = pd.DataFrame(
    reverse_checks
)

display(reverse_checks)

,protocol_1,protocol_2,direction_difference
0,AB → BC → AC,AC → BC → AB,6.284760e-10
1,AB → AC → BC,BC → AC → AB,1.640808e-09
2,BC → AB → AC,AC → AB → BC,6.036730e-10


In [67]:
unique_protocols = [
    "AB → BC → AC",
    "AB → AC → BC",
    "BC → AB → AC"
]


unique_directions = []

for protocol in unique_protocols:

    row = protocol_direction_results[
        protocol_direction_results[
            "protocol"
        ] == protocol
    ].iloc[0]

    unique_directions.append([
        row["coefficient_C_AB"],
        row["coefficient_C_BC"]
    ])


unique_directions = np.array(
    unique_directions
)


print(
    "Sum of the three protocol directions:",
    unique_directions.sum(axis=0)
)

Sum of the three protocol directions: [ 0.       -0.000003]


In [68]:
protocol_direction_results.to_csv(
    result_dir / "six_protocol_cycle_directions.csv",
    index=False
)

reverse_checks.to_csv(
    result_dir / "reverse_protocol_checks.csv",
    index=False
)